In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import json

In [2]:
# Load customer data
customer_df = pd.read_csv('customers_updated2.csv')

# Load transaction data
transaction_df = pd.read_excel('final_transact.xlsx')

print(customer_df.head())
print(transaction_df.head())

transaction_df['Transaction Details'] = transaction_df['Transaction Details'].str.replace('-', ' ', regex=True)

# Display cleaned data
print(transaction_df.head())

             Full Name  Account Number  CIBIL Score  \
0          Renee Dayal    794843871722          367   
1          Bhavin Toor    963450436285          601   
2  Nitara Venkataraman    314939397126          678   
3           Purab Lata    269085222633          427   
4           Nitara Lad    801266852126          824   

                                   Address           State      City Country  \
0  138, Dugar Path, Rajpur Sonarpur-590544   Uttar Pradesh   Lucknow   India   
1              368, Chana, Amravati-727268  Madhya Pradesh    Indore   India   
2        10/81, Dhar Road, Dehradun-400493       Rajasthan    Jaipur   India   
3      54, Chhabra Nagar, Darbhanga-266575     West Bengal  Siliguri   India   
4   H.No. 784, Kadakia Zila, Ratlam-780493          Punjab  Ludhiana   India   

   Mobile Number PAN Card Number           Occupation  \
0     6990527801      UTVQH01965           Pharmacist   
1     6598772407      QWYJQ44260  Marketing Executive   
2     6188847212 

In [3]:
label_encoder = LabelEncoder()
customer_df['Occupation'] = label_encoder.fit_transform(customer_df['Occupation'])
customer_df['Gender'] = label_encoder.fit_transform(customer_df['Gender'])

In [4]:
customer_df['Monthly_Income'] = customer_df['Annual Income (in Rs)'] / 12

In [5]:
def calculate_credit_limit(row):
    if row['CIBIL Score'] >= 800:
        x = 5
    elif row['CIBIL Score'] >= 700:
        x = 3
    elif row['CIBIL Score'] >= 600:
        x = 2
    else:
        x = 1
    return row['Monthly_Income'] * x

customer_df['Credit_Limit'] = customer_df.apply(calculate_credit_limit, axis=1)

In [6]:
def categorize_transaction(transaction):
    transaction = transaction.lower()
    
    # Dining
    if any(keyword in transaction for keyword in [
        'restaurant', 'dining', 'zomato', 'swiggy', 
        "mcdonald's", "domino's", "starbucks", "kfc", "pizza hut", "burger king"
    ]):
        return 'Dining'
    
    # Travel 
    elif any(keyword in transaction for keyword in [
        'travel', 'airbnb', 'hotel', 'makemytrip', 'uber', 'booking.com', 'expedia'
    ]):
        return 'Travel'
    
    # Fuel 
    elif any(keyword in transaction for keyword in [
        'fuel', 'shell', 'hpcl', 'indianoil', 'tesla', 'chevron', 'exxon', 'bp'
    ]):
        return 'Fuel'
    
    # Shopping 
    elif any(keyword in transaction for keyword in [
        'shopping', 'zara', 'myntra', 'flipkart', 'amazon', 'costco', 
        'walmart', 'h&m', 'nike', 'adidas', 'target', 'gap', 'uniqlo'
    ]):
        return 'Shopping'
    
    # Grocery 
    elif any(keyword in transaction for keyword in [
        'grocery', 'bigbasket', 'grofers', 'dmart', 'whole foods', 'kroger'
    ]):
        return 'Grocery'
    
    # Subscription 
    elif any(keyword in transaction for keyword in [
        'subscription', 'netflix', 'spotify', 'apple music', 'prime video', 'hulu', 'disney+'
    ]):
        return 'Subscription'
    
    # Foreign 
    elif any(keyword in transaction for keyword in [
        'foreign', 'forex', 'usd', 'uk', 'germany', 'japan', 
        'usa', 'australia', 'canada', 'euro', 'singapore'
    ]):
        return 'Foreign'
    
    # Finance category
    elif any(keyword in transaction for keyword in [
        'finance', 'paypal', 'bank transfer', 'credit card', 'debit card'
    ]):
        return 'Finance'
    
    # Social Media category
    elif any(keyword in transaction for keyword in [
        'social', 'facebook', 'twitter', 'instagram', 'snapchat', 'linkedin', 'tiktok'
    ]):
        return 'Social Media'
    
    # Health category
    elif any(keyword in transaction for keyword in [
        'health', 'hospital', 'pharmacy', 'clinic', 'medicines', 'insurance'
    ]):
        return 'Health'
    
    # Education category
    elif any(keyword in transaction for keyword in [
        'education', 'school', 'college', 'university', 'course', 'learning'
    ]):
        return 'Education'
    
    # Utility category
    elif any(keyword in transaction for keyword in [
        'electricity', 'water', 'gas', 'internet', 'mobile', 'recharge', 'bill'
    ]):
        return 'Utility'
    
    # Mixed category
    elif 'mixed' in transaction:
        return 'Mixed'
    
    else:
        return 'Other'
transaction_df['Category'] = transaction_df['Transaction Details'].apply(categorize_transaction)


In [7]:
spending_patterns = transaction_df.groupby(['Account Number', 'Category'])['Withdrawal Amount'].sum().unstack(fill_value=0)
spending_patterns.reset_index(inplace=True)

In [8]:
data = pd.merge(customer_df, spending_patterns, on='Account Number', how='left')
data.fillna(0, inplace=True)


In [9]:
card_criteria = {
    'Union JCB Wellness Credit Card': {'min_income': 25000, 'min_score': 700, 'categories': ['Dining', 'Shopping']},
    'RuPay Select Credit Card': {'min_income': 25000, 'min_score': 720, 'categories': ['Travel', 'Dining']},
    'Union JCB Health Credit Card': {'min_income': 30000, 'min_score': 700, 'categories': ['Dining', 'Shopping']},
    'Union UNI-CARBON Credit Card': {'min_income': 50000, 'min_score': 700, 'categories': ['Fuel', 'Shopping']},
    'RuPay Platinum Credit Card': {'min_income': 30000, 'min_score': 700, 'categories': ['Dining', 'Travel']},
    'VISA Signature Credit Card': {'min_income': 75000, 'min_score': 750, 'categories': ['Travel', 'Shopping']},
    'VISA Platinum Credit Card': {'min_income': 50000, 'min_score': 700, 'categories': ['Fuel', 'Dining']},
    'VISA Gold Credit Card': {'min_income': 25000, 'min_score': 700, 'categories': ['Shopping', 'Dining']},
    'Divaā Credit Card': {'min_income': 25000, 'min_score': 700, 'categories': ['Shopping', 'Dining']}
}

In [10]:
def recommend_cards(row):
    recommendations = []
    
    for card, criteria in card_criteria.items():
        if 'min_income' in criteria and row['Monthly_Income'] < criteria['min_income']:
            continue
        if 'min_score' in criteria and row['CIBIL Score'] < criteria['min_score']:
            continue
        if 'categories' in criteria:
            matched_categories = [cat for cat in criteria['categories'] if row.get(cat, 0) > 0]
            if not matched_categories:
                continue
        
        recommendations.append(card)
    
    return recommendations

data['Recommended_Cards'] = data.apply(recommend_cards, axis=1)

In [11]:
missing_cols = ['Travel', 'Dining', 'Shopping', 'Fuel', 'Grocery', 'Subscription', 'Foreign']
for col in missing_cols:
    if col not in data.columns:
        data[col] = 0

In [12]:
features = data[['Monthly_Income', 'CIBIL Score', 'Travel', 'Dining', 'Shopping', 'Fuel', 'Grocery', 'Subscription', 'Foreign']]
target = data['Recommended_Cards'].apply(lambda x: x[0] if x else 'No Card')

# Label encode target values
target = label_encoder.fit_transform(target)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [14]:
output = data[['Full Name', 'Account Number', 'Recommended_Cards']].to_dict(orient='records')

# Save to JSON file
with open('recommendations.json', 'w') as f:
    json.dump(output, f, indent=4)

# Display output
print(json.dumps(output, indent=4))


[
    {
        "Full Name": "Renee Dayal",
        "Account Number": 794843871722,
        "Recommended_Cards": []
    },
    {
        "Full Name": "Bhavin Toor",
        "Account Number": 963450436285,
        "Recommended_Cards": []
    },
    {
        "Full Name": "Nitara Venkataraman",
        "Account Number": 314939397126,
        "Recommended_Cards": []
    },
    {
        "Full Name": "Purab Lata",
        "Account Number": 269085222633,
        "Recommended_Cards": []
    },
    {
        "Full Name": "Nitara Lad",
        "Account Number": 801266852126,
        "Recommended_Cards": [
            "Union JCB Wellness Credit Card",
            "RuPay Select Credit Card",
            "Union JCB Health Credit Card",
            "RuPay Platinum Credit Card",
            "VISA Gold Credit Card",
            "Diva\u0101 Credit Card"
        ]
    },
    {
        "Full Name": "Lavanya Sankar",
        "Account Number": 528072251252,
        "Recommended_Cards": []
    },
    {
  